# 09 — Rotary positions and correct cached decoding

RoPE rotates pairs of query/key coordinates by position-dependent angles. For an adjacent pair:
$R(\theta)(a,b)=(a\cos\theta-b\sin\theta,\;a\sin\theta+b\cos\theta)$.

This lab uses adjacent pairs and base 10,000. Real checkpoints may use a different coordinate layout or frequency configuration. We test geometry and cache correctness, not long-context quality.

## How to work through this notebook

Run setup once. At each checkpoint, write a prediction and try the small implementation before reading its adjacent reference solution. All reference cells run unchanged from top to bottom; exercise cells contain safe, optional starting points. Numerical checks use CPU float64 unless explicitly noted. Agent-verified reference execution is separate from your learning progress.

In [ ]:
from pathlib import Path
import sys, copy, math, inspect
from dataclasses import replace
import torch
from torch import nn
from torch.nn import functional as F
root = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "src/dongxi_llms/decoder_lab.py").exists()), None)
if root is None:
    raise RuntimeError("Open this notebook from inside the Dongxi_LLMs repository")
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))
from dongxi_llms.decoder_lab import (
    DecoderConfig, TinyDecoder, DecoderBlock, MultiHeadAttention, MLP, RMSNorm,
    layer_norm, rms_norm, rope, attend, parameter_count, analytical_parameters,
    cost_estimate, teaching_batch, next_token_loss, fit_one_batch)
torch.set_num_threads(1)
torch.manual_seed(505)
DTYPE = torch.float64
def close(actual, expected, atol=1e-10, rtol=1e-8):
    torch.testing.assert_close(actual, expected, atol=atol, rtol=rtol)
print("CPU reference environment:", torch.__version__)


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from IPython.display import display
from dongxi_llms import decoder_visuals as viz
def show_visual(figure):
    display(figure)
    plt.close(figure)


## Architecture map — your location in the model

The highlighted stage is this lesson’s focus. B = batch, T = positions, D = model width, V = vocabulary size. This is a structural map, not measured activations or runtime. The modern route uses RoPE inside attention rather than adding learned absolute positions.

![Architecture map — your location in the model. The highlighted stage is this lesson’s focus. B = batch, T = positions, D = model width, V = vocabulary size. This is a structural map, not measured activations or runtime. The modern route uses RoPE inside attention rather than adding learned absolute positions.](../figures/chapter-05/day-06-02_rotary_positions-architecture-map.png)

*Saved architecture schematic. The following cell regenerates it; it does not execute or train a model.*

In [ ]:
from dongxi_llms import decoder_architecture as architecture
show_visual(architecture.model_map(focus='positions', modern=True))

## Place rotation on the correct branches

RoPE transforms Q and K using their positions. V bypasses rotation. The diagram matches this lesson's model, where QK normalization is disabled; the next lesson adds that optional operation.

![Place rotation on the correct branches. RoPE transforms Q and K using their positions. V bypasses rotation. The diagram matches this lesson's model, where QK normalization is disabled; the next lesson adds that optional operation.](../figures/chapter-05/day-06-02_rotary_positions-architecture-detail.png)

*Saved architecture schematic. The following cell regenerates it; it does not execute or train a model.*

In [ ]:
show_visual(architecture.rope_detail())

## 1. Rotate coordinate pairs

Implement the rotation explicitly. What should happen to each pair's squared length?

**Your prediction:** _Write it here before running the reference._

In [ ]:
q = torch.randn(1, 2, 4, 4, dtype=DTYPE, requires_grad=True)
positions = torch.arange(4)
# Your implementation: pair rotations using cos/sin.

### Reference solution

Run after your attempt; compare the mechanism, not just the final numbers.

In [ ]:
inv_frequency = 10000. ** (-torch.arange(0, 4, 2, dtype=DTYPE)/4)
angles = positions[:, None]*inv_frequency
even, odd = q[..., 0::2], q[..., 1::2]
manual = torch.stack((even*angles.cos()-odd*angles.sin(),
                      even*angles.sin()+odd*angles.cos()), -1).flatten(-2)
close(manual, rope(q, positions))
close(manual.reshape(1, 2, 4, 2, 2).square().sum(-1),
      q.reshape(1, 2, 4, 2, 2).square().sum(-1))
print("Before/after first head:", q[0, 0].detach(), manual[0, 0].detach())
assert torch.autograd.gradcheck(lambda z: rope(z, positions), (q,))
print("Pairwise norm and numerical derivative checks passed.")

### Why this works

A rotation preserves pair length while changing direction. Position zero gives the identity rotation. The frequency differs across pairs, so not every feature pair turns by the same angle.

### Visual explanation — See position as a rotation

These arrows show the first coordinate pair at positions 0 and 1 of head 0. The circle is a constant-radius guide: position 0 leaves the pair unchanged, while position 1 rotates it.

The figure uses this lesson’s tensors. Rerun it after changing the preceding experiment.

![See position as a rotation. These arrows show the first coordinate pair at positions 0 and 1 of head 0. The circle is a constant-radius guide: position 0 leaves the pair unchanged, while position 1 rotates it.](../figures/chapter-05/day-06-02_rotary_positions-visual-rope-geometry.png)

*Saved reference preview. The code below regenerates this figure from the current lesson state; it does not overwrite the preview.*

In [ ]:
show_visual(viz.rotary(q[0,0], manual[0,0], positions))

## 2. Derive relative-position compatibility

Check (R_m q) dot (R_n k) = q dot (R_(n-m) k). Then shift both absolute positions by the same amount.

**Your prediction:** _Write it here before running the reference._

In [ ]:
k = torch.randn_like(q)
m, n = torch.arange(4)+3, torch.arange(4)+8
# Your implementation: compare the dot products.

### Reference solution

Run after your attempt; compare the mechanism, not just the final numbers.

In [ ]:
left = (rope(q, m)*rope(k, n)).sum(-1)
relative = (q*rope(k, n-m)).sum(-1)
shifted = (rope(q, m+20)*rope(k, n+20)).sum(-1)
close(left, relative)
close(left, shifted)
print("Relative-position identity error:", float((left-relative).abs().max().detach()))

### Why this works

The identity follows from R_m transpose times R_n = R_(n-m). This holds for fixed unrotated q/k vectors. Hidden states in an actual model also depend on context, so it does not make all model behavior depend only on a single relative distance.

## 3. Replay a modern decoder with a cache

Prefill two tokens, then add tokens one at a time. Compare every new output with full-sequence causal outputs.

**Your prediction:** _Write it here before running the reference._

In [ ]:
cfg = DecoderConfig(modern=True, kv_heads=2)
model = TinyDecoder(cfg).double().eval()
ids, _ = teaching_batch()
# Your implementation: retain one compact KV pair per layer.

### Reference solution

Run after your attempt; compare the mechanism, not just the final numbers.

In [ ]:
with torch.no_grad():
    full = model(ids)
    first, cache = model(ids[:, :2], return_cache=True)
    parts = [first]
    for t in range(2, ids.shape[1]):
        output, cache = model(ids[:, t:t+1], caches=cache, return_cache=True)
        parts.append(output)
    replay = torch.cat(parts, 1)
close(replay, full)
print("Cache replay max error:", float((replay-full).abs().max()))
print("One layer's compact K shape:", cache[0][0].shape)

### Why this works

Cached keys retain their original rotations. New queries and keys use continuing absolute positions; cached keys are not rotated again. During a decode step, all cached positions through the current token are valid sources.

## 4. Restart the position offset incorrectly

Keep the correct causal visibility but rotate the new chunk as if it started at position zero. Is the attention mask enough to protect correctness?

**Your prediction:** _Write it here before running the reference._

In [ ]:
# Predict whether this error is future leakage or incorrect positional geometry.

### Reference solution

Run after your attempt; compare the mechanism, not just the final numbers.

In [ ]:
with torch.no_grad():
    _, prefix_cache = model(ids[:, :2], return_cache=True)
    broken = model(ids[:, 2:], caches=prefix_cache, rope_offset=0)
error = float((broken-full[:, 2:]).abs().max())
assert error > 1e-8
print("Wrong-offset cache error:", error)

### Why this works

The mask can remain causal while the result is wrong. Old keys and new queries now disagree about positions. This is a positional-consistency error, not necessarily future leakage. Cache correctness requires both causal boundaries and consistent position handling.

### Visual explanation — See the cache-offset error

The same full-prefix suffix is the reference for both panels. Correct cached replay differs only by floating-point rounding; restarting positions creates a larger error without removing the causal mask.

The figure uses this lesson’s tensors. Rerun it after changing the preceding experiment.

![See the cache-offset error. The same full-prefix suffix is the reference for both panels. Correct cached replay differs only by floating-point rounding; restarting positions creates a larger error without removing the causal mask.](../figures/chapter-05/day-06-02_rotary_positions-visual-cache-offset.png)

*Saved reference preview. The code below regenerates this figure from the current lesson state; it does not overwrite the preview.*

In [ ]:
show_visual(viz.matrices([(replay-full)[0,2:], (broken-full[:,2:])[0]], ['Correct cached replay: error', 'Wrong position offset: error'], 'Position consistency is part of cache correctness', xlabel='Candidate token ID', ylabel='Suffix position'))

## Takeaway and evidence boundary

Next: reduce the number of stored key/value heads and account explicitly for what changes—and what does not.

Companion map: [Chapter 5 pathway](../day-05/README.md). Reusable source: [decoder_lab.py](../../src/dongxi_llms/decoder_lab.py). Record your explanation and remaining questions here; the notebook's existence does not mark the lesson complete.